# 🌾 AgriVision Agent: Deep Learning Transfer Learning & Agentic RAG
### College Project: Artificial Neural Networks (ANN) & Deep Learning

**Authors:** AgriVision AI Development Team  
**Core Architecture:** MobileNetV2 Transfer Learning + LangGraph Agentic Workflow + FAISS RAG  
**Dataset:** PlantVillage Multi-Crop Disease Benchmark  

---

## 1. Mathematical Foundations of MobileNetV2 Transfer Learning

Standard Convolutional Neural Networks perform spatial and channel-wise computations simultaneously with standard 2D convolution kernels of size $D_K \times D_K \times M \times N$, resulting in computational cost:
$$\text{Cost}_{\text{std}} = D_K \cdot D_K \cdot M \cdot N \cdot D_F \cdot D_F$$

MobileNetV2 decomposes this into **Depthwise Separable Convolutions** with **Inverted Residual Bottlenecks** and **Linear Bottlenecks**:
1. **Depthwise Convolution:** Applies a single filter per input channel ($D_K \times D_K \times 1 \times M$).
2. **Pointwise Convolution:** Computes linear combinations of the output channels using $1 \times 1$ convolutions ($1 \times 1 \times M \times N$).

The reduced computational complexity is:
$$\text{Cost}_{\text{MobileNet}} = D_K \cdot D_K \cdot M \cdot D_F \cdot D_F + M \cdot N \cdot D_F \cdot D_F$$
$$\text{Reduction Factor} = \frac{1}{N} + \frac{1}{D_K^2} \approx \frac{1}{9} \text{ to } \frac{1}{8} \text{ computation of standard CNNs.}$$

---

In [ ]:
# Step 1: Import Core Libraries
import sys
from pathlib import Path
import numpy as np
import matplotlib.pyplot as plt
import tensorflow as tf
from PIL import Image

# Add project root
PROJECT_ROOT = Path("..").resolve()
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.config import config
from src.utils.dataset import load_class_indices, get_available_samples
from src.utils.image_processing import load_and_validate_image, apply_clahe_enhancement, segment_leaf_mask
from src.models.model_builder import build_transfer_learning_model
from src.models.inference import get_classifier
from src.rag.knowledge_base import get_structured_prescriptions, query_rag
from src.agent.graph import run_agent_workflow

print("TensorFlow Version:", tf.__version__)
print("GPU Devices Available:", tf.config.list_physical_devices('GPU'))

## 2. Computer Vision Preprocessing & Foliage Segmentation
Evaluating CLAHE contrast enhancement and morphological segmentation.

In [ ]:
# Load sample benchmark leaf
samples = get_available_samples()
sample_path = list(samples.values())[0]
raw_img = load_and_validate_image(sample_path)

enhanced = apply_clahe_enhancement(raw_img)
seg = segment_leaf_mask(raw_img)

fig, axes = plt.subplots(1, 3, figsize=(12, 4))
axes[0].imshow(raw_img)
axes[0].set_title("Original Leaf RGB")
axes[0].axis("off")

axes[1].imshow(enhanced)
axes[1].set_title("CLAHE Enhanced")
axes[1].axis("off")

axes[2].imshow(seg["segmented_image"])
axes[2].set_title(f"Segmented (Necrotic Area: {seg['lesion_percentage']}%)")
axes[2].axis("off")

plt.tight_layout()
plt.show()

## 3. Deep Learning Model Inference & Top-3 Probabilities

In [ ]:
classifier = get_classifier()
pred = classifier.predict(raw_img, compute_cam=True)

print(f"Predicted Crop: {pred['crop']}")
print(f"Predicted Condition: {pred['disease']}")
print(f"Confidence: {pred['confidence_percentage']}%")
print(f"Severity Level: {pred['severity_level']}")
print("\nTop-3 Probabilities:")
for t in pred["top3_predictions"]:
    print(f" - {t['full_name']}: {t['confidence_pct']}")

## 4. Grad-CAM Neural Attention Heatmap Visualization

In [ ]:
if pred.get("gradcam_overlay") is not None:
    plt.figure(figsize=(6, 6))
    plt.imshow(pred["gradcam_overlay"])
    plt.title(f"Grad-CAM Activation Map ({pred['crop']} - {pred['disease']})")
    plt.axis("off")
    plt.show()

## 5. LangGraph Autonomous Agent Reasoning & RAG Retrieval

In [ ]:
# Execute multi-node agent workflow
agent_res = run_agent_workflow(
    crop=pred["crop"],
    disease=pred["disease"],
    confidence=pred["confidence"],
    field_acres=2.0,
    location="Nagpur, India",
    language="en"
)

print("=== AGENT REASONING STEPS ===")
for s in agent_res["reasoning_steps"]:
    print(" ->", s)

print("\n=== FORMULATED PRESCRIPTION ===")
rx = agent_res["final_prescription"]
print("Biological Controls:", rx.get("biological_controls", []))
print("Chemical Controls:", rx.get("chemical_controls", []))
print("Field Dosage:", rx.get("field_dosage", {}))
print("Hindi Advisory:", rx.get("hindi_summary", ""))